Notebook creates & saves basic hypothesis sets for PRC, plus performs wald test. 

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

2025-10-29 10:59:06.953781: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-29 10:59:06.958141: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
from pathlib import Path

In [3]:
DATA_ROOT=Path("/gpfs/gibbs/pi/reilly/tabula_data")
path=DATA_ROOT/"simulated"
name="shendure_calibrated_sim_with_orthos_20251008"

In [4]:
demo_counts=scm.scMPRA_data.from_parquet(path/name/"scMPRA/0.scmpra")
demo_counts.ortho_filter()

scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [5]:
# all CREs within each cell type, vs the 'reference' negative control
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=demo_counts,
    reference_cre="reference",
    meta="emvar_screen",
)

# all cell types for each CRE, vs the dataset’s baseline cell type
hs_all_cre = scm.make_all_by_cre_hypotheses(
    counts=demo_counts,
    reference_cell_type="reference",  # will be normalized to 'reference'
    meta="cell_specificity",
)

In [6]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=8,#cores per slurm job
        memory="80G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p ycga", 
            f"--job-name=simclust_worker",
            f"--time=1:00:00",
            f"--output=slave_%j.out"]
    )
    cluster.scale(jobs=3)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

In [7]:
from dask.distributed import Semaphore, as_completed, get_client

In [8]:
ortho_root=path/name/"orthos_with_precomputed_wald_erin_test"
output_root=path/name/"results"
output_root.mkdir(exist_ok=True,parents=True)
input_ortho_names=[path.name for path in ortho_root.iterdir()]

Semaphore(max_leases=3, name="test")

def compute_one_wald(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type):
    sem = Semaphore(name="test")
    with sem:
        client=get_client()
        ortho_oi=scm.ortho.load(client=client,
                                    path=input_root,
                                    name=name)
        runner = scm.HypothesisTester(test_type)
        output_short=Path(output_root)/hypothesis_set_name/test_type
        output_short.mkdir(exist_ok=True,parents=True)
        runner.run(hypothesis_set, ortho_oi, client).to_tsv(output_short/name)



In [9]:
#hs_all_ct
futures = [client.submit(compute_one_wald,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_ct,
                        hypothesis_set_name="hs_all_ct",
                        test_type="wald") for name_oi in input_ortho_names]

In [10]:
#hs_all_cre
futures = [client.submit(compute_one_wald,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_cre,
                        hypothesis_set_name="hs_all_cre",
                        test_type="wald") for name_oi in input_ortho_names]

In [9]:
scmpradat_root=path/name/"scMPRA"
def compute_one_mwu(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type):
    sem = Semaphore(name="test")
    with sem:
        client=get_client()
        dat=scm.scMPRA_data.from_parquet(scmpradat_root/Path(name).with_suffix(".scmpra"))
        dat.ortho_filter()
        runner = scm.HypothesisTester(test_type)
        output_short=Path(output_root)/hypothesis_set_name/test_type
        output_short.mkdir(exist_ok=True,parents=True)
        runner.run(hypothesis_set, dat, client).to_tsv(output_short/name)

In [12]:
futures_mwu = [client.submit(compute_one_mwu,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_ct,
                        hypothesis_set_name="hs_all_ct",
                        test_type="mwu") for name_oi in input_ortho_names]

In [10]:
futures_mwu = [client.submit(compute_one_mwu,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=hs_all_cre,
                        hypothesis_set_name="hs_all_cre",
                        test_type="mwu") for name_oi in input_ortho_names]

In [11]:
client.close()
cluster.close()